# The open-world hole, live

`SHACL` is the default RDF validator, and it is **open-world**: it validates the terms a shape targets and stays silent about everything else. So when a language model emits a *plausible-but-nonexistent* ontology term, SHACL reports `conforms = true` and the fabricated data flows on, looking exactly like clean data. A **closed-world vocabulary gate** catches it.

**Press `Runtime -> Run all`** (or the play button on each cell). It installs two packages, then runs both validators on three preloaded hallucinated examples so you can see the contrast. The last cell is yours to edit.

Full benchmark and reproducible code: https://github.com/fabio-rovai/open-ontologies/tree/main/case-studies/onto-correctness-bench


In [ ]:
!pip -q install rdflib pyshacl requests


In [ ]:
import json, requests
import rdflib
from rdflib import Graph, RDF, URIRef, Literal
from rdflib.namespace import SH
from pyshacl import validate

RAW = "https://raw.githubusercontent.com/fabio-rovai/open-ontologies/main/case-studies/onto-correctness-bench/space/vocab"
VOCAB = {}
for key, label in [("schemaorg","schema.org"), ("ies4","IES4"), ("obo","OBO (PATO+RO)")]:
    d = requests.get(f"{RAW}/{key}.json", timeout=60).json()
    VOCAB[label] = {"policed": d["policed"], "declared": set(d["declared"])}
print("loaded vocabularies:", ", ".join(f"{k} ({len(v['declared'])} terms)" for k,v in VOCAB.items()))

def in_policed(iri, policed):
    return any(str(iri).startswith(p) for p in policed)

def closed_world_flags(g, policed, declared):
    flagged = set()
    for s,p,o in g:
        if in_policed(p, policed) and str(p) not in declared: flagged.add(str(p))
        if p == RDF.type and isinstance(o, URIRef) and in_policed(o, policed) and str(o) not in declared: flagged.add(str(o))
    return sorted(flagged)

def build_shapes(g, policed, declared):
    shapes = Graph(); n = 0
    for subj in set(g.subjects(RDF.type, None)):
        for cls in g.objects(subj, RDF.type):
            if not (isinstance(cls, URIRef) and in_policed(cls, policed) and str(cls) in declared): continue
            shape = URIRef(f"https://example.org/shape/{n}"); n += 1
            shapes.add((shape, RDF.type, SH.NodeShape)); shapes.add((shape, SH.targetClass, cls))
            for p in set(g.predicates(subj, None)):
                if in_policed(p, policed) and str(p) in declared:
                    b = URIRef(f"https://example.org/shape/{n}/p"); n += 1
                    shapes.add((shape, SH.property, b)); shapes.add((b, SH.path, p)); shapes.add((b, SH.minCount, Literal(1)))
    return shapes

def check(turtle, vocab_label):
    v = VOCAB[vocab_label]
    g = Graph().parse(data=turtle, format="turtle")
    shapes = build_shapes(g, v["policed"], v["declared"])
    conforms, _, _ = validate(g, shacl_graph=shapes, inference="none", abort_on_first=False)
    flags = closed_world_flags(g, v["policed"], v["declared"])
    print(f"VOCABULARY: {vocab_label}")
    print(f"  SHACL (open-world):   conforms = {str(conforms).lower()}   <- {'admits the fabricated term(s)' if conforms else 'violation found'}")
    if flags:
        print(f"  Closed-world gate:    REJECTED, {len(flags)} undeclared term(s):")
        for f in flags: print(f"      - {f}")
    else:
        print("  Closed-world gate:    PASSED, every term is declared")
    print()


In [ ]:
SCHEMA_ORG = """@prefix schema: <https://schema.org/> .
@prefix ex: <https://example.org/> .
# schema:MerchandiseOffer (class) and schema:priceBracket (predicate) do NOT exist.
ex:offer1 a schema:Offer , schema:MerchandiseOffer ;
    schema:priceCurrency "GBP" ; schema:price "49.00" ; schema:priceBracket "mid" ."""

IES4 = """@prefix ies: <http://ies.data.gov.uk/ontology/ies4#> .
@prefix ex: <https://example.org/> .
# ies:hasParticipant does not exist; the real term is ies:isParticipantIn.
ex:event1 a ies:Event ; ies:isParticipationOf ex:person1 ; ies:hasParticipant ex:person1 ."""

OBO = """@prefix obo: <http://purl.obolibrary.org/obo/> .
@prefix ex: <https://example.org/> .
# obo:PATO_9999999 is a well-formed but undeclared PATO id.
ex:x1 a obo:PATO_0000462 ; obo:RO_0000052 ex:thing1 ; a obo:PATO_9999999 ."""

check(SCHEMA_ORG, "schema.org")
check(IES4, "IES4")
check(OBO, "OBO (PATO+RO)")


## Your turn

Edit the Turtle below and re-run this cell. Use `schema.org`, `IES4`, or `OBO (PATO+RO)` as the vocabulary.


In [ ]:
MY_RDF = """@prefix schema: <https://schema.org/> .
@prefix ex: <https://example.org/> .
ex:thing a schema:Product ;
    schema:name "Widget" ;
    schema:colour "blue" ."""   # schema uses schema:color, not schema:colour -> watch the gate catch it

check(MY_RDF, "schema.org")


---
Measured across three real vocabularies, open-world SHACL passed **300/300** graphs carrying a fabricated term; the closed-world gate caught **300/300** with **0** false positives. Read the write-up: https://gov.tesseract.academy/research/ontology-correctness-benchmark

Built by Tesseract Academy.
